# Hands-On Data Lab Implementation
**Author:** Nayan Deepak Borse

This notebook sets up a data analysis environment and completes practical exercises in data manipulation, visualization and basic analysis using a logistics shipments dataset.

## 1. Environment setup
Install the libraries once from a terminal:

```
pip install jupyter pandas numpy matplotlib seaborn
jupyter notebook
```

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

print("Python:", sys.version.split()[0])
for lib in (np, pd, matplotlib, sns):
    print(lib.__name__, lib.__version__)
sns.set_theme(style="whitegrid")

## 2. Create the dataset
A reproducible sample of 1,000 shipments with a few missing values added on purpose.

In [ ]:
rng = np.random.default_rng(42)
n = 1000
regions = ["North", "South", "East", "West", "Central"]
carriers = ["FastTrack", "SwiftMove", "CargoPlus", "RoadKing"]

df = pd.DataFrame({
    "shipment_id": np.arange(1, n + 1),
    "order_date": pd.to_datetime("2025-01-01") + pd.to_timedelta(rng.integers(0, 365, n), unit="D"),
    "region": rng.choice(regions, n),
    "carrier": rng.choice(carriers, n),
    "distance_km": rng.integers(50, 2000, n).astype(float),
    "weight_kg": rng.gamma(4, 120, n).round(1),
})
df["delivery_days"] = (df["distance_km"] / 400 + rng.normal(2, 0.8, n)).clip(1).round(1)
df["cost_inr"] = (df["distance_km"] * 6 + df["weight_kg"] * 4 + rng.normal(0, 500, n)).clip(lower=300).round(0)

# add missing values
df.loc[rng.choice(n, 40, replace=False), "weight_kg"] = np.nan
df.loc[rng.choice(n, 25, replace=False), "delivery_days"] = np.nan
df.head()

## 3. Data manipulation exercises

In [ ]:
# Exercise 1: structure and missing values
df.info()
print(df.isnull().sum())

In [ ]:
# Exercise 2: handle missing values
df["weight_kg"] = df["weight_kg"].fillna(df["weight_kg"].median())
df["delivery_days"] = df["delivery_days"].fillna(df.groupby("carrier")["delivery_days"].transform("median"))
print("Missing values left:", df.isnull().sum().sum())

In [ ]:
# Exercise 3: filtering and new columns
heavy = df[(df["weight_kg"] > 800) & (df["distance_km"] > 1000)]
print("Heavy long distance shipments:", len(heavy))

df["cost_per_km"] = (df["cost_inr"] / df["distance_km"]).round(2)
df["month"] = df["order_date"].dt.month
df["on_time"] = np.where(df["delivery_days"] <= 4, "Yes", "No")
df.head()

In [ ]:
# Exercise 4: groupby and aggregation
summary = df.groupby("carrier").agg(
    shipments=("shipment_id", "count"),
    avg_cost=("cost_inr", "mean"),
    avg_days=("delivery_days", "mean"),
).round(2).sort_values("avg_days")
summary

In [ ]:
# Exercise 5: merge with a second table
carrier_info = pd.DataFrame({
    "carrier": carriers,
    "fleet_size": [120, 80, 150, 60],
})
merged = df.merge(carrier_info, on="carrier", how="left")
merged[["shipment_id", "carrier", "fleet_size"]].head()

## 4. Visualization exercises

In [ ]:
# Bar chart: average delivery days by region
df.groupby("region")["delivery_days"].mean().sort_values().plot(kind="bar", color="#2E75B6")
plt.title("Average Delivery Days by Region")
plt.ylabel("Days")
plt.tight_layout()
plt.savefig("chart_bar_region.png", dpi=150)
plt.show()

In [ ]:
# Line chart: monthly shipments
df.groupby("month").size().plot(marker="o", color="#C55A11")
plt.title("Monthly Shipment Volume")
plt.xlabel("Month")
plt.ylabel("Shipments")
plt.tight_layout()
plt.savefig("chart_line_monthly.png", dpi=150)
plt.show()

In [ ]:
# Histogram: distribution of cost
sns.histplot(df["cost_inr"], bins=30, kde=True, color="#548235")
plt.title("Distribution of Shipment Cost (INR)")
plt.tight_layout()
plt.savefig("chart_hist_cost.png", dpi=150)
plt.show()

In [ ]:
# Heatmap: correlation of numeric columns
num = df[["distance_km", "weight_kg", "delivery_days", "cost_inr"]]
sns.heatmap(num.corr(), annot=True, cmap="Blues", fmt=".2f")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.savefig("chart_heatmap.png", dpi=150)
plt.show()

## 5. Basic analysis

In [ ]:
print(num.describe().round(2))
print()
print("Correlation (distance vs cost):", round(df["distance_km"].corr(df["cost_inr"]), 3))
print("Correlation (distance vs delivery days):", round(df["distance_km"].corr(df["delivery_days"]), 3))
print()
print("On time rate by carrier (%):")
print((df.assign(ot=df["on_time"].eq("Yes")).groupby("carrier")["ot"].mean() * 100).round(1))

## 6. Conclusion
- The environment runs with Python, Jupyter, pandas, NumPy, Matplotlib and Seaborn.
- Missing values were filled with median based methods.
- Distance is the main driver of cost and delivery time.
- Carrier level summaries help compare speed and cost.